In [1]:
%pip install prophet

  Using cached matplotlib-3.10.7-cp313-cp313-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp313-cp313-macosx_10_13_universal2.whl.metadata (112 kB)
  Using cached kiwisolver-1.4.9-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pillow-12.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 49.6 MB/s  0:00:00.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.9 MB/s  0:00:00
Using cached matplotlib-3.10.7-cp313-cp313-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl (274 

### 1. Segment dataset

In [ ]:
import pandas as pd
# load & prep
reg_cols = [
    "platt_apag_mom_pct", "platt_crude_mom_pct",
    "temp_avg", "precip_avg",
    "threshold_temp_days", "threshold_rainy_days",
    "possible_working_days", "avg_weight"
]

df = pd.read_csv("../data/final_features_3.csv")
df["ds"] = pd.to_datetime(df["date"], format="%Y%m") + pd.offsets.MonthEnd(0)
df = df.rename(columns={"demand": "y"})
df[reg_cols] = df[reg_cols].astype(float)

cutoff = df["ds"].max() - pd.DateOffset(months=32)
df_train = df[df["ds"] <= cutoff].copy()
df_valid = df[df["ds"] > cutoff].copy()

### 2. Prophet baseline

In [10]:
from prophet import Prophet


# fit
m = Prophet(yearly_seasonality=True)
for reg in reg_cols:
    m.add_regressor(reg)
m.fit(df_train)

# build future frame and attach regressors
future = m.make_future_dataframe(periods=12, freq="M")
future = future.merge(df[["ds"] + reg_cols], on="ds", how="left")

# supply regressor values for the forecast horizon (replace with your own logic)
for col in reg_cols:
    future[col] = future[col].fillna(df[col].iloc[-12:].mean())

forecast = m.predict(future)
forecast.to_csv("../data/forecast_output.csv", index=False)


12:33:32 - cmdstanpy - INFO - Chain [1] start processing
12:33:32 - cmdstanpy - INFO - Chain [1] done processing
/Users/chungsangyoon/Desktop/Playground/ml_project/.venv/lib/python3.13/site-packages/prophet/forecaster.py:1872: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
